# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Client:
 Version:    29.8.1
 Context:    default
 Debug Mode: false
 Plugins:
  agent: Docker AI Agent Runner (Docker Inc.)
    Version:  v1.141.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-agent
  ai: Docker AI Agent - Ask Gordon (Docker Inc.)
    Version:  v1.31.0
    Path:     /usr/local/lib/docker/cli-plugins/docker-ai
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /usr/local/lib/docker/cli-plugins/docker-compose
  debug: Get a shell into any image or container (Docker Inc.)
    Version:  0.0.47
    Path:     /usr/local/lib/docker/cli-plugins/docker-debug
  desktop: Docker Desktop commands (Docker Inc.)
    Version:  v0.4.4
    Path:     /usr/local/lib/docker/cli-plugins/docker-desktop
  dhi: CLI for managing Docker Hardened Images (Docker Inc.)
    Version:  v0.0.7
    Path:     /usr/local/lib/docker/cli-plugins/docker-dh

### What is a container?

A container is an isolated environment that runs software together with its required dependencies.

### Why do we use containers?

Containers simplify software setup, avoid dependency conflicts and help make analyses reproducible across computers.

### What is a docker image?

A Docker image is a template containing software, dependencies and settings. Docker uses it to create containers.

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
(workflows) zwc@DESKTOP-45NJ8KT:~/computational-workflows-2026$ docker login
Authenticating with existing credentials... [Username: maomaoliqwq]

i Info → To login with a different account, run 'docker logout' followed by 'docker login'


Login Succeeded

### Run your first docker container

In [2]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



### Find the container ID

In [3]:
!docker ps -a --filter "name=day04-hello"

CONTAINER ID   IMAGE         COMMAND    CREATED         STATUS                     PORTS     NAMES
82e518ea45cc   hello-world   "/hello"   6 minutes ago   Exited (0) 6 minutes ago             day04-hello


### Delete the container again, give prove its deleted

In [4]:
!docker rm day04-hello

day04-hello


In [5]:
!docker ps -a --filter "name=day04-hello"

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

In [8]:
!fastqc --version

FastQC v0.12.1


Please describe the steps you took to download and run the software for the example fastq file from last week below:

In [ ]:
fastqc \
  --threads 4 \
  --outdir fastqc_direct_results \
  SRX19144486_SRR23195516_1.fastq.gz

I installed FastQC in my conda environment and checked its version. I then ran FastQC on one FASTQ file from the previous exercise (SRX19144486_SRR23195516_1.fastq.gz) and saved the HTML report and ZIP file in fastqc_direct_results.

### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [ ]:
# pull the container
FASTQC_IMAGE='community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae'

docker pull "$FASTQC_IMAGE"

cd ~/computational-workflows-2026/day_04
mkdir -p fastqc_results
# run the container and save the results to a new "fastqc_results" directory
docker run --rm \
  --user "$(id -u):$(id -g)" \
  -v "$PWD:/data" \
  -w /data \
  "$FASTQC_IMAGE" \
  fastqc --threads 4 \
  --outdir fastqc_results \
  SRX19144486_SRR23195516_1.fastq.gz

i used diff \
  <(unzip -p fastqc_direct_results/*_fastqc.zip '*/fastqc_data.txt') \
  <(unzip -p fastqc_results/*_fastqc.zip '*/fastqc_data.txt') to compare the results and they are the same.

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

Using an existing Docker image was straightforward and avoided installing FastQC and its dependencies locally. Once downloaded, the image can be reused for future analyses.

### What would you say, which approach is more reproducible?

The Docker approach is generally more reproducible because it packages the software and its dependencies together. Recording the exact image identifier helps others use the same environment.

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

Both runs used FastQC version 0.12.1 and produced identical QC data for the same FASTQ file.

## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor

### Explain the RUN and ENV lines you added to the file

In [ ]:
(workflows) zwc@DESKTOP-45NJ8KT:~/computational-workflows-2026/day_04$ docker run --rm day04-cowsay:1.0 cowsay "Hello from my Docker image!"
 _____________________________
< Hello from my Docker image! >
 -----------------------------
        \   ^__^
         \  (oo)\_______
            (__)\       )\/\
                ||----w |
                ||     ||

In [ ]:
# build the docker image
docker build --pull --no-cache -f my_dockerfile -t day04-cowsay:1.0 .

In [ ]:
# make sure that the image has been built


In [ ]:
# run the docker file 
docker run --rm day04-cowsay:1.0 cowsay "Hello from my Docker image!"

## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image

In [ ]:
# build the image
docker build -f salmon_docker -t day04-salmon:1.5.2 .

In [ ]:
# run the docker image to give out the version of salmon
docker run --rm day04-salmon:1.5.2 salmon --version

## Do you think bioinformaticians have to create a docker image every time they want to run a tool?

Find the salmon docker image online and run it on your computer.

What is https://biocontainers.pro/ ?

In [ ]:
                                                                                            i Info →   U  In Use
IMAGE                ID             DISK USAGE   CONTENT SIZE   EXTRA
day04-salmon:1.5.2   e683997b5f82        409MB         88.5MB
                                                                                            i Info →   U  In Use
IMAGE                     ID             DISK USAGE   CONTENT SIZE   EXTRA
combinelab/salmon:1.5.2   1d5c7b736c01       2.12GB          570MB

Both images run Salmon 1.5.2. My custom image uses less disk space. The difference in size comes from the base image and included dependencies. Using a prebuilt image is convenient because I do not need to install Salmon myself.

BioContainers provides ready-to-use containers for bioinformatics tools. These containers include the software and its dependencies, making installation easier and helping reproduce analyses.

## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

Container images can be built manually using a Dockerfile, downloaded from a container registry, or generated using services such as Seqera Containers.  
Seqera Containers lets users select software packages and versions and generates a container image with the required dependencies. I used it to obtain a FastQC 0.12.1 image without writing a Dockerfile. This makes software setup easier and helps create reproducible environments.